In [0]:
from pyspark.sql import functions as F

df_gold = spark.table("teste_koin.default.gold_orders_customers")

df_customer_metrics = (
    df_gold
    .groupBy(
        "customer_id",
        "customer_name",
        "customer_email_hash",
        "customer_city",
        "customer_state"
    )
    .agg(
        F.count("order_id").alias("total_orders"),
        F.round(F.sum("order_amount"), 2).alias("total_spent"),
        F.round(F.avg("order_amount"), 2).alias("average_ticket"),
        F.max("order_date").alias("last_order_date"),
        F.min("order_date").alias("first_order_date")
    )
)

(
    df_customer_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("teste_koin.default.gold_customer_metrics")
)